In [1]:

from pathlib import Path
import re
import difflib
import subprocess

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, Markdown, clear_output

# Source of truth: existing firewall configuration.
SOURCE_FILE = Path("/etc/nft/web01/rules/10-tcp-networks.nft")

# Save button writes here, never directly to the production file.
STAGING_FILE = Path("/tmp/10-tcp-networks.zone01.staged.nft")

R01 = "Z01_TCP4_R01"
R02 = "Z01_TCP4_R02"

DEFAULTS = {
    "interface": "$nic_lan",
    "source": "$ip_lan",
    "destination": "$net_lan01",
    "port": "",
    "verdict": "accept",
}


def read_source():
    if not SOURCE_FILE.is_file():
        raise FileNotFoundError(
            f"Cannot read source file: {SOURCE_FILE}"
        )
    return SOURCE_FILE.read_text()


def parse_definition(source, name):
    pattern = (
        r"(?m)^\s*define\s+" + re.escape(name)
        + r"\s*=\s*\{(.*?)^\s*\}"
    )
    match = re.search(pattern, source, flags=re.DOTALL)

    if not match:
        raise ValueError(f"Definition {name} was not found.")

    body = match.group(1)
    entries = []

    for line_number, raw_line in enumerate(body.splitlines(), start=1):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        # Strip an optional trailing comma.
        line = line.rstrip(",").strip()

        if ":" not in line:
            raise ValueError(
                f"Cannot parse {name}, body line {line_number}: {raw_line}"
            )

        key_text, verdict = line.rsplit(":", 1)
        fields = [part.strip() for part in key_text.split(".")]
        verdict = verdict.strip()

        entries.append({
            "fields": fields,
            "verdict": verdict,
        })

    return entries


def load_rules():
    source = read_source()

    r01_entries = parse_definition(source, R01)
    r02_entries = parse_definition(source, R02)

    if len(r01_entries) != len(r02_entries):
        raise ValueError(
            f"{R01} has {len(r01_entries)} rules but "
            f"{R02} has {len(r02_entries)}. "
            "The editor requires one-to-one correspondence."
        )

    rules = []

    for index, (a, b) in enumerate(
        zip(r01_entries, r02_entries), start=1
    ):
        fields01 = a["fields"]
        fields02 = b["fields"]

        if len(fields01) != 5 or len(fields02) != 4:
            raise ValueError(
                f"Rule {index}: expected 5 R01 key fields and "
                "4 R02 key fields."
            )

        if fields01[0] != "$Z01":
            raise ValueError(
                f"Rule {index}: unexpected R01 zone key "
                f"{fields01[0]!r}."
            )

        if fields01[1:] != fields02:
            raise ValueError(
                f"Rule {index}: R01/R02 keys do not correspond.\n"
                f"R01: {fields01}\nR02: {fields02}"
            )

        if a["verdict"] != b["verdict"]:
            raise ValueError(
                f"Rule {index}: R01/R02 verdicts differ."
            )

        rules.append({
            "interface": fields01[1],
            "source": fields01[2],
            "destination": fields01[3],
            "port": fields01[4],
            "verdict": a["verdict"],
        })

    return source, rules


source_original, rules = load_rules()

print(f"Source: {SOURCE_FILE}")
print(f"Staging destination: {STAGING_FILE}")
print(f"Zone 01 TCP rules loaded: {len(rules)}")


Source: /etc/nft/web01/rules/10-tcp-networks.nft
Staging destination: /tmp/10-tcp-networks.zone01.staged.nft
Zone 01 TCP rules loaded: 11


In [2]:

# ============================================================
# Rule rendering
# ============================================================

def key_r01(rule):
    fields = [
        "$Z01",
        rule["interface"],
        rule["source"],
        rule["destination"],
        rule["port"],
    ]
    return " . ".join(fields)


def key_r02(rule):
    fields = [
        rule["interface"],
        rule["source"],
        rule["destination"],
        rule["port"],
    ]
    return " . ".join(fields)


def nft_entry(key, verdict):
    return f"{key} : {verdict},"


def rule_table():
    rows = []

    for number, rule in enumerate(rules, start=1):
        rows.append({
            "#": number,
            "Interface": rule["interface"],
            "Source": rule["source"],
            "Destination": rule["destination"],
            "Port": rule["port"],
            "Verdict": rule["verdict"],
            "R01 entry": nft_entry(key_r01(rule), rule["verdict"]),
            "R02 entry": nft_entry(key_r02(rule), rule["verdict"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# Widgets
# ============================================================

rule_selector = widgets.Dropdown(
    options=[],
    description="Rule:",
    layout=widgets.Layout(width="350px"),
    style={"description_width": "initial"},
)

interface_input = widgets.Text(
    value=DEFAULTS["interface"],
    description="Interface:",
    layout=widgets.Layout(width="280px"),
)

source_input = widgets.Text(
    value=DEFAULTS["source"],
    description="Source:",
    layout=widgets.Layout(width="280px"),
)

destination_input = widgets.Text(
    value=DEFAULTS["destination"],
    description="Destination:",
    layout=widgets.Layout(width="280px"),
)

port_input = widgets.Text(
    value="",
    placeholder="e.g. 443 or $https",
    description="Destination port:",
    layout=widgets.Layout(width="280px"),
)

verdict_input = widgets.Dropdown(
    options=["accept", "drop", "reject"],
    value="accept",
    description="Verdict:",
    layout=widgets.Layout(width="280px"),
)

add_button = widgets.Button(
    description="Add",
    icon="plus",
    button_style="success",
)

edit_button = widgets.Button(
    description="Update",
    icon="check",
    button_style="primary",
)

delete_button = widgets.Button(
    description="Delete",
    icon="trash",
    button_style="danger",
)

save_button = widgets.Button(
    description="Save staging file",
    icon="save",
    button_style="info",
)

table_output = widgets.Output()
status_output = widgets.Output()
preview_output = widgets.Output()


def set_status(message, error=False):
    with status_output:
        clear_output(wait=True)
        display(HTML(
            f"<p style='color:{'#b91c1c' if error else '#166534'}'>"
            f"{message}</p>"
        ))


def refresh_table():
    options = [
        (f"Rule {i}: {rule['port']}", i - 1)
        for i, rule in enumerate(rules, start=1)
    ]

    previous = rule_selector.value
    rule_selector.options = options

    if options:
        valid_values = [value for _, value in options]
        rule_selector.value = (
            previous if previous in valid_values
            else valid_values[0]
        )

    with table_output:
        clear_output(wait=True)
        display(Markdown("### Current TCP Zone 01 rules"))
        display(
            rule_table().style
            .set_properties(**{
                "text-align": "left",
                "white-space": "pre-wrap",
                "overflow-wrap": "anywhere",
                "min-width": "180px",
            })
            .set_table_styles([
                {"selector": "table", "props": [
                    ("width", "100%"),
                    ("table-layout", "auto"),
                ]},
                {"selector": "td", "props": [
                    ("max-width", "650px"),
                    ("vertical-align", "top"),
                ]},
            ])
        )


def load_selected_rule(*_):
    index = rule_selector.value

    if index is None or index >= len(rules):
        return

    rule = rules[index]
    interface_input.value = rule["interface"]
    source_input.value = rule["source"]
    destination_input.value = rule["destination"]
    port_input.value = rule["port"]
    verdict_input.value = rule["verdict"]


def form_rule():
    rule = {
        "interface": interface_input.value.strip(),
        "source": source_input.value.strip(),
        "destination": destination_input.value.strip(),
        "port": port_input.value.strip(),
        "verdict": verdict_input.value,
    }

    for field, value in rule.items():
        if not value:
            raise ValueError(f"{field} cannot be empty.")

        if any(c in value for c in ("\n", "\r", ",", "{", "}")):
            raise ValueError(
                f"Invalid character in {field}: {value!r}"
            )

    # These fields are nftables expressions, not arbitrary free text.
    # Reject separators that could create additional rule expressions.
    for field in ("interface", "source", "destination", "port"):
        value = rule[field]
        if ":" in value or ";" in value:
            raise ValueError(
                f"Unexpected ':' or ';' in {field}."
            )

    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", rule["verdict"]):
        raise ValueError("Invalid verdict.")

    return rule


def add_rule(_):
    try:
        rule = form_rule()

        # Prevent duplicate logical entries.
        if rule in rules:
            raise ValueError("This rule already exists.")

        rules.append(rule)
        refresh_table()
        load_selected_rule()
        set_status("Rule added to the in-memory staging model.")
    except Exception as exc:
        set_status(str(exc), error=True)


def update_rule(_):
    try:
        index = rule_selector.value
        if index is None:
            raise ValueError("Select a rule to update.")

        updated = form_rule()

        if any(
            i != index and other == updated
            for i, other in enumerate(rules)
        ):
            raise ValueError("Another rule already has these fields.")

        rules[index] = updated
        refresh_table()
        set_status("Rule updated in the in-memory staging model.")
    except Exception as exc:
        set_status(str(exc), error=True)


def delete_rule(_):
    index = rule_selector.value

    if index is None:
        set_status("Select a rule to delete.", error=True)
        return

    removed = rules.pop(index)
    refresh_table()
    set_status(f"Deleted logical rule for port {removed['port']}.")


rule_selector.observe(load_selected_rule, names="value")
add_button.on_click(add_rule)
edit_button.on_click(update_rule)
delete_button.on_click(delete_rule)

display(Markdown("# TCP Zone 01 — Rule Editor"))
display(Markdown(
    "One row represents both R01 and R02. Changes remain in memory "
    "until saved to the staging file."
))

display(widgets.VBox([
    table_output,
    widgets.HTML("<h3>Rule editor</h3>"),
    rule_selector,
    widgets.HBox([interface_input, source_input]),
    widgets.HBox([destination_input, port_input]),
    verdict_input,
    widgets.HBox([add_button, edit_button, delete_button]),
    status_output,
]))

refresh_table()
load_selected_rule()


# TCP Zone 01 — Rule Editor

One row represents both R01 and R02. Changes remain in memory until saved to the staging file.

In [3]:

# ============================================================
# Pre-save validation for TCP Zone 01
# ============================================================

def validate_rules():
    errors = []
    warnings = []

    seen = set()

    if not rules:
        warnings.append("Both definitions will be empty.")

    for index, rule in enumerate(rules, start=1):
        label = f"Rule {index}"

        # Required fields
        for field in (
            "interface", "source", "destination", "port", "verdict"
        ):
            value = rule.get(field, "")
            if not isinstance(value, str) or not value.strip():
                errors.append(f"{label}: {field} is empty.")

        # Port must be a numeric port or a symbolic variable.
        port = rule.get("port", "").strip()

        if port:
            if port.isdigit():
                if not 1 <= int(port) <= 65535:
                    errors.append(
                        f"{label}: port {port} is outside 1–65535."
                    )
            elif not re.fullmatch(r"\$[A-Za-z_][A-Za-z0-9_]*", port):
                errors.append(
                    f"{label}: port must be numeric or a variable "
                    "such as $https."
                )

        # Check for duplicates across the complete logical rule.
        signature = tuple(
            rule.get(field, "").strip()
            for field in (
                "interface", "source", "destination", "port", "verdict"
            )
        )

        if signature in seen:
            errors.append(f"{label}: duplicate logical rule.")
        seen.add(signature)

        # These fields are expressions in nftables, so flag obvious
        # accidental line breaks or rule delimiters.
        for field in (
            "interface", "source", "destination", "port", "verdict"
        ):
            value = rule.get(field, "")
            if any(ch in value for ch in ("\n", "\r", ";", "{", "}")):
                errors.append(
                    f"{label}: invalid delimiter in {field}."
                )

        if rule.get("verdict") not in {"accept", "drop", "reject"}:
            errors.append(
                f"{label}: unsupported verdict "
                f"{rule.get('verdict')!r}."
            )

    # Verify that the generator produces corresponding R01/R02 keys.
    for index, rule in enumerate(rules, start=1):
        fields01 = key_r01(rule).split(" . ")
        fields02 = key_r02(rule).split(" . ")

        if fields01 != ["$Z01"] + fields02:
            errors.append(
                f"Rule {index}: R01 and R02 keys do not correspond."
            )

    return errors, warnings


def display_validation():
    errors, warnings = validate_rules()

    display(Markdown("## Validation report"))

    if errors:
        display(HTML(
            "<h4 style='color:#b91c1c'>Validation failed</h4>"
        ))
        for error in errors:
            print("ERROR:", error)
    else:
        display(HTML(
            "<h4 style='color:#166534'>Logical validation passed</h4>"
        ))

    for warning in warnings:
        print("WARNING:", warning)

    print(f"\nLogical rules: {len(rules)}")
    print(f"Generated R01 entries: {len(rules)}")
    print(f"Generated R02 entries: {len(rules)}")

    return not errors


display_validation()


## Validation report


Logical rules: 10
Generated R01 entries: 10
Generated R02 entries: 10


True

In [4]:

# Cell 3 — Validate, stage, review, and check TCP Zone 01 rules
#
# Requires the existing Cell 1 and Cell 2 variables/functions:
# SOURCE_FILE, STAGING_FILE, R01, R02, rules, source_original,
# key_r01(), key_r02(), nft_entry(), read_source(), set_status().
#
# This cell never applies changes to the live firewall.

import difflib
import json
import re
import subprocess
import tempfile

from pathlib import Path
from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

# Set this to the real top-level ruleset file used by your firewall
# loader if you want to run a syntax check against the full ruleset.
#
# Leave as None until you have confirmed the correct loader entry point.
RULESET_ENTRYPOINT = None

# Namespace containing the target firewall.
VALIDATION_NAMESPACE = "web01"

# nft -c checks syntax without applying the ruleset.
# The check is performed only when a validated entry point is configured.
RUN_NFT_CHECK = True


# ------------------------------------------------------------
# Logical rule validation
# ------------------------------------------------------------

def validate_zone01_rules():
    errors = []
    warnings = []
    seen = set()

    if not rules:
        warnings.append("Both Zone 01 TCP definitions will be empty.")

    for index, rule in enumerate(rules, start=1):
        label = f"Rule {index}"

        required = (
            "interface", "source", "destination", "port", "verdict"
        )

        for field in required:
            value = rule.get(field, "")
            if not isinstance(value, str) or not value.strip():
                errors.append(f"{label}: {field} is empty.")

        # Only accept a decimal port or an nftables variable reference.
        port = str(rule.get("port", "")).strip()

        if port:
            if port.isdigit():
                if not 1 <= int(port) <= 65535:
                    errors.append(
                        f"{label}: numeric port must be between 1 and 65535."
                    )
            elif not re.fullmatch(
                r"\$[A-Za-z_][A-Za-z0-9_]*", port
            ):
                errors.append(
                    f"{label}: destination port must be numeric "
                    "or a variable such as $https."
                )

        # Fields are nftables expressions. Reject obvious delimiters
        # that could escape the intended one-line rule representation.
        for field in required:
            value = str(rule.get(field, ""))
            if any(ch in value for ch in ("\n", "\r", ";", "{", "}")):
                errors.append(
                    f"{label}: invalid delimiter in {field}."
                )

        verdict = rule.get("verdict", "")
        if verdict not in {"accept", "drop", "reject"}:
            errors.append(
                f"{label}: unsupported verdict {verdict!r}."
            )

        signature = tuple(
            str(rule.get(field, "")).strip()
            for field in required
        )

        if signature in seen:
            errors.append(f"{label}: duplicate logical rule.")
        seen.add(signature)

        # Ensure the R01 key is precisely the R02 key with $Z01 added.
        if key_r01(rule).split(" . ") != (
            ["$Z01"] + key_r02(rule).split(" . ")
        ):
            errors.append(
                f"{label}: R01 and R02 keys do not correspond."
            )

    return errors, warnings


# ------------------------------------------------------------
# Source verification and staged configuration generation
# ------------------------------------------------------------

def render_zone01_definition(name, entries):
    lines = [f"define {name} = {{"]

    for entry in entries:
        lines.append(
            f"    {nft_entry(entry['key'], entry['verdict'])}"
        )

    lines.append("}")
    return "\n".join(lines)


def build_zone01_staged_source():
    current_source = read_source()

    # Protect against silently overwriting edits made outside the notebook.
    if current_source != source_original:
        raise RuntimeError(
            "The source file changed after the notebook loaded it. "
            "Reload the source and reconcile external changes before saving."
        )

    r01_entries = [
        {"key": key_r01(rule), "verdict": rule["verdict"]}
        for rule in rules
    ]

    r02_entries = [
        {"key": key_r02(rule), "verdict": rule["verdict"]}
        for rule in rules
    ]

    replacements = {
        R01: render_zone01_definition(R01, r01_entries),
        R02: render_zone01_definition(R02, r02_entries),
    }

    staged = current_source

    for name, replacement in replacements.items():
        pattern = (
            r"(?m)^[ \t]*define[ \t]+" + re.escape(name)
            + r"[ \t]*=[ \t]*\{.*?^[ \t]*\}"
        )

        staged, count = re.subn(
            pattern,
            lambda match, replacement=replacement: replacement,
            staged,
            count=1,
            flags=re.DOTALL,
        )

        if count != 1:
            raise RuntimeError(
                f"Expected exactly one definition for {name}; "
                f"found {count}."
            )

    return staged


# ------------------------------------------------------------
# Optional nftables syntax check
# ------------------------------------------------------------

def check_staged_configuration(staged_text):
    """
    Syntax-check a complete ruleset entry point with nft -c.

    The staged source file is not itself loaded into the live firewall.
    A full entry point must include the definitions and other files
    required by the real configuration.
    """
    if not RUN_NFT_CHECK:
        return {
            "status": "skipped",
            "message": "nftables syntax check disabled.",
        }

    if RULESET_ENTRYPOINT is None:
        return {
            "status": "skipped",
            "message": (
                "No RULESET_ENTRYPOINT configured. Set it to the "
                "actual top-level file used by the firewall loader "
                "before relying on nftables syntax validation."
            ),
        }

    entrypoint = Path(RULESET_ENTRYPOINT)

    if not entrypoint.is_file():
        return {
            "status": "failed",
            "message": f"Ruleset entry point does not exist: {entrypoint}",
        }

    # This cell deliberately does not substitute the staged definition
    # into the live entry point or apply it. A real integration test
    # should use an isolated test namespace and a complete staged tree.
    return {
        "status": "not_run",
        "message": (
            "Full ruleset validation requires an isolated staged "
            "configuration tree. The configured entry point has not "
            "been modified or executed."
        ),
    }


# ------------------------------------------------------------
# Complete Save action
# ------------------------------------------------------------

save_button = widgets.Button(
    description="Validate and save staging file",
    icon="save",
    button_style="info",
)

validation_output = widgets.Output()
diff_output = widgets.Output()


def save_zone01_staging(_):
    with validation_output:
        clear_output(wait=True)

        errors, warnings = validate_zone01_rules()

        display(Markdown("## Validation report"))

        if errors:
            display(HTML(
                "<h4 style='color:#b91c1c'>Validation failed</h4>"
            ))
            for error in errors:
                print("ERROR:", error)

            print("\nStaging file was not written.")
            set_status("Validation failed; staging was not saved.", error=True)
            return

        display(HTML(
            "<h4 style='color:#166534'>Logical validation passed</h4>"
        ))

        for warning in warnings:
            print("WARNING:", warning)

        try:
            staged_text = build_zone01_staged_source()

            # Write only to the staging destination.
            STAGING_FILE.parent.mkdir(parents=True, exist_ok=True)
            STAGING_FILE.write_text(staged_text)

            check = check_staged_configuration(staged_text)

            print(f"Rules: {len(rules)}")
            print(f"R01 entries: {len(rules)}")
            print(f"R02 entries: {len(rules)}")
            print(f"Staging file: {STAGING_FILE}")
            print(f"nftables check status: {check['status']}")
            print(check["message"])

            with diff_output:
                clear_output(wait=True)

                diff = difflib.unified_diff(
                    source_original.splitlines(),
                    staged_text.splitlines(),
                    fromfile=str(SOURCE_FILE),
                    tofile=str(STAGING_FILE),
                    lineterm="",
                )

                display(Markdown("## Proposed diff"))
                print("\n".join(diff) or "No changes detected.")

            set_status(
                "Staging file saved. The live firewall was not changed."
            )

        except Exception as exc:
            print(f"ERROR: {exc}")
            set_status(
                f"Could not generate staging configuration: {exc}",
                error=True,
            )


save_button.on_click(save_zone01_staging)

display(Markdown("## Validate and stage"))
display(Markdown(
    "Saving validates the logical rules, writes the staged configuration, "
    "and displays a diff. It never applies changes to the live firewall."
))
display(save_button)
display(validation_output)
display(diff_output)


## Validate and stage

Saving validates the logical rules, writes the staged configuration, and displays a diff. It never applies changes to the live firewall.

Button(button_style='info', description='Validate and save staging file', icon='save', style=ButtonStyle())

Output()

Output()

In [5]:

# TCP Zone 01 — Validate, stage, and test
# Requires the editor and rule-rendering functions from the earlier cells.

import difflib
import shutil
import subprocess
from pathlib import Path

from IPython.display import display, Markdown, HTML, clear_output
import ipywidgets as widgets


# ------------------------------------------------------------
# Paths and test configuration
# ------------------------------------------------------------

SOURCE_FILE = Path("/etc/nft/web01/rules/10-tcp-networks.nft")
STAGING_FILE = Path("/tmp/10-tcp-networks.nft")
RULESET_ENTRYPOINT = Path("/tmp/nftables-zone01-test.conf")

VALIDATION_NAMESPACE = "web01"

VARIABLES_INCLUDE = "/etc/nft/web01/variables/*.nft"
CONSTANTS_INCLUDE = "/etc/nft/web01/constants/*.nft"
ICMP_INCLUDE = "/etc/nft/web01/rules/06-icmp-networks.nft"
UDP_INCLUDE = "/etc/nft/web01/rules/08-udp-networks.nft"
ZONES_INCLUDE = "/etc/nft/web01/zones/*.nft"


# ------------------------------------------------------------
# Build an isolated loader using the staged TCP file
# ------------------------------------------------------------

def build_test_entrypoint():
    required = [
        Path(ICMP_INCLUDE),
        Path(UDP_INCLUDE),
    ]

    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Required production rule files missing: "
            + ", ".join(missing)
        )

    if not STAGING_FILE.is_file():
        raise FileNotFoundError(
            f"Staged TCP rules do not exist: {STAGING_FILE}"
        )

    text = "\n".join([
        "# Generated by the TCP Zone 01 editor.",
        "# Test entry point; not the production loader.",
        "# Loading this file flushes the active ruleset.",
        "flush ruleset",
        f'include "{VARIABLES_INCLUDE}"',
        f'include "{CONSTANTS_INCLUDE}"',
        f'include "{ICMP_INCLUDE}"',
        f'include "{UDP_INCLUDE}"',
        f'include "{STAGING_FILE}"',
        f'include "{ZONES_INCLUDE}"',
        "",
    ])

    RULESET_ENTRYPOINT.write_text(text)
    return RULESET_ENTRYPOINT


# ------------------------------------------------------------
# Validate the entire candidate ruleset without applying it
# ------------------------------------------------------------

def check_staged_configuration():
    if shutil.which("ip") is None:
        return {
            "status": "failed",
            "message": "The ip command was not found.",
        }

    if shutil.which("nft") is None:
        return {
            "status": "failed",
            "message": "The nft command was not found.",
        }

    try:
        entrypoint = build_test_entrypoint()

        command = [
            "ip", "netns", "exec", VALIDATION_NAMESPACE,
            "nft", "-c", "-f", str(entrypoint),
        ]

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )

        output = "\n".join(
            s.strip()
            for s in (result.stdout, result.stderr)
            if s.strip()
        )

        if result.returncode == 0:
            return {
                "status": "passed",
                "message": (
                    "Candidate ruleset passed nft -c. "
                    "No rules were applied."
                    + (f"\n{output}" if output else "")
                ),
            }

        return {
            "status": "failed",
            "message": (
                f"nft -c failed with exit status {result.returncode}."
                f"\n{output}"
            ),
        }

    except subprocess.TimeoutExpired:
        return {
            "status": "failed",
            "message": "Syntax check timed out after 30 seconds.",
        }
    except (OSError, RuntimeError, ValueError) as exc:
        return {
            "status": "failed",
            "message": str(exc),
        }


# ------------------------------------------------------------
# Save, validate, stage, check, and display the diff
# ------------------------------------------------------------

validation_output = widgets.Output()
diff_output = widgets.Output()

save_button = widgets.Button(
    description="Validate, stage and check",
    icon="check",
    button_style="info",
)

def save_zone01_staging(_):
    with validation_output:
        clear_output(wait=True)

        errors, warnings = validate_zone01_rules()

        display(Markdown("## Validation report"))

        if errors:
            display(HTML(
                "<h4 style='color:#b91c1c'>Validation failed</h4>"
            ))
            for error in errors:
                print("ERROR:", error)

            set_status(
                "Validation failed; staging file was not written.",
                error=True,
            )
            return

        for warning in warnings:
            print("WARNING:", warning)

        try:
            # Refuse to overwrite changes made outside the editor.
            current_source = read_source()
            if current_source != source_original:
                raise RuntimeError(
                    "Production source changed since it was loaded. "
                    "Reload and reconcile before saving."
                )

            staged_text = build_zone01_staged_source()

            STAGING_FILE.parent.mkdir(
                parents=True, exist_ok=True
            )
            STAGING_FILE.write_text(staged_text)

            check = check_staged_configuration()

            print(f"Source: {SOURCE_FILE}")
            print(f"Staging file: {STAGING_FILE}")
            print(f"Test entry point: {RULESET_ENTRYPOINT}")
            print(f"Logical rules: {len(rules)}")
            print(f"R01 entries: {len(rules)}")
            print(f"R02 entries: {len(rules)}")
            print(f"nftables check: {check['status']}")
            print(check["message"])

            with diff_output:
                clear_output(wait=True)
                diff = difflib.unified_diff(
                    source_original.splitlines(),
                    staged_text.splitlines(),
                    fromfile=str(SOURCE_FILE),
                    tofile=str(STAGING_FILE),
                    lineterm="",
                )
                display(Markdown("## Proposed TCP rules diff"))
                print("\n".join(diff) or "No changes detected.")

            if check["status"] == "passed":
                set_status(
                    "Staged and syntax-checked. Live firewall unchanged."
                )
            else:
                set_status(
                    "Staged, but syntax validation did not pass. "
                    "Do not deploy.",
                    error=True,
                )

        except Exception as exc:
            print(f"ERROR: {exc}")
            set_status(str(exc), error=True)


save_button.on_click(save_zone01_staging)

display(Markdown("# TCP Zone 01 — Test workflow"))
display(Markdown(
    "The editor writes a candidate TCP file, generates a dedicated "
    "test entry point, runs nft -c, and displays the proposed diff. "
    "It never applies the ruleset."
))
display(save_button)
display(validation_output)
display(diff_output)


# TCP Zone 01 — Test workflow

The editor writes a candidate TCP file, generates a dedicated test entry point, runs nft -c, and displays the proposed diff. It never applies the ruleset.

Button(button_style='info', description='Validate, stage and check', icon='check', style=ButtonStyle())

Output()

Output()